In [1]:
from pyspark.sql import SparkSession

# packages for iceberg + spark + kafka
PACKAGES = [
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "org.postgresql:postgresql:42.6.0",
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
]

spark = SparkSession.builder \
    .appName("Iceberg_Setup") \
    .config("spark.jars.packages", ",".join(PACKAGES)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog") \
    .config("spark.sql.catalog.my_catalog.uri", "jdbc:postgresql://postgres:5432/iceberg_metastore") \
    .config("spark.sql.catalog.my_catalog.jdbc.user", "iceberg") \
    .config("spark.sql.catalog.my_catalog.jdbc.password", "iceberg") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
print('123')

123


In [4]:
import time
from datetime import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
from pyspark.sql.functions import col, from_json

file_thresholds = [101, 133, 250, 500] 
run_duration_sec = 2 * 60 * 60 
table_name = "my_catalog.default.user_events_file_based"

# json schema definition
json_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("shop", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("session_id", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

def clean_environment(phase_name):
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")
    
    spark.sql(f"""
    CREATE TABLE {table_name} (
        user_id INT,
        shop STRING,    
        event_type STRING,
        device_type STRING,
        price DOUBLE,
        session_id STRING,
        timestamp TIMESTAMP
    )
    USING iceberg
    PARTITIONED BY (hours(timestamp))
    TBLPROPERTIES (
        'format-version'='2',
        'write.format.default'='parquet',
        'write.target-file-size-bytes'='30720'
    )
    """)
    print(f"[{datetime.now()}] environment cleaned for phase: {phase_name}")
for threshold in file_thresholds:
    phase_label = f"{threshold}_files_threshold"
    print(f"\n================ starting phase: {phase_label} ================")
    
    clean_environment(phase_label)
    
    unique_run_id = int(time.time())
    checkpoint_path = f"s3a://warehouse/checkpoints/snapshot_based_{phase_label}_{unique_run_id}"
    
    kafka_stream = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "kafka:9092") \
        .option("subscribe", "user_events_bursty") \
        .option("startingOffsets", "latest") \
        .load()

    parsed_stream = kafka_stream \
        .selectExpr("CAST(value AS STRING) as json_string") \
        .select(from_json(col("json_string"), json_schema).alias("data")) \
        .select("data.*")
    
    streaming_query = parsed_stream.writeStream \
        .format("iceberg") \
        .outputMode("append") \
        .trigger(processingTime="5 seconds") \
        .option("checkpointLocation", checkpoint_path) \
        .toTable(table_name)
        
    print(f"stream active for {phase_label}. beginning execution timer.")
    
    start_phase_time = time.time()
    
    while time.time() - start_phase_time < run_duration_sec:
        try:
            current_files_count = spark.sql(f"SELECT COUNT(*) FROM {table_name}.files").collect()[0][0]
        except Exception as e:
            current_files_count = 0
            
        print(f"[{datetime.now()}] current files: {current_files_count} / threshold: {threshold}")
        
        if current_files_count >= threshold:
            print(f"[{datetime.now()}] threshold reached! triggering compaction...")
            
            compaction_start = time.time()
            
            spark.sql(f"""
                CALL my_catalog.system.rewrite_data_files(
                    table => '{table_name.replace("my_catalog.", "")}',
                    strategy => 'sort',
                    sort_order => 'zorder(shop, device_type, event_type)',
                    options => map(
                        'partial-progress.enabled', 'true',
                        'partial-progress.max-commits', '10',
                        'min-input-files', '2',
                        'target-file-size-bytes', '4194304',
                        'rewrite-all', 'true'
                    )
                )
            """)
            
            compaction_duration = time.time() - compaction_start
            print(f"[{datetime.now()}] compaction cycle complete in {compaction_duration:.2f} seconds.")
            
            # даємо системі трохи більше часу на "відпочинок" після важкого запиту
            time.sleep(30) 
        else:
            # якщо ліміт не досягнуто, перевіряємо частіше
            time.sleep(15)
        
    print(f"phase {phase_label} completed. terminating stream cleanly...")
    streaming_query.stop()
    streaming_query.awaitTermination()

print("all automated benchmarking phases executed successfully.")


================ starting phase: 101_files_threshold ================
[2026-05-23 22:13:29.396927] environment cleaned for phase: 101_files_threshold
stream active for 101_files_threshold. beginning execution timer.
[2026-05-23 22:13:29.701238] current files: 0 / threshold: 101
[2026-05-23 22:13:44.792233] current files: 4 / threshold: 101
[2026-05-23 22:13:59.904692] current files: 7 / threshold: 101
[2026-05-23 22:14:15.043929] current files: 10 / threshold: 101
[2026-05-23 22:14:30.314567] current files: 13 / threshold: 101
[2026-05-23 22:14:45.417752] current files: 16 / threshold: 101
[2026-05-23 22:15:00.534546] current files: 20 / threshold: 101
[2026-05-23 22:15:15.637794] current files: 23 / threshold: 101
[2026-05-23 22:15:30.746712] current files: 26 / threshold: 101
[2026-05-23 22:15:45.869456] current files: 29 / threshold: 101
[2026-05-23 22:16:00.999796] current files: 32 / threshold: 101
[2026-05-23 22:16:16.119403] current files: 35 / threshold: 101
[2026-05-23 22:16: